In [1]:
from kafka import KafkaProducer
import json
import random
import time
from datetime import datetime, timedelta
import uuid

# ---------------------------------------------------
# KAFKA PRODUCER
# ---------------------------------------------------

producer = KafkaProducer(
    bootstrap_servers='localhost:9092',
    value_serializer=lambda v:
    json.dumps(v).encode('utf-8')
)

print("Producer Started...\n")

# ---------------------------------------------------
# CONFIG
# ---------------------------------------------------

TOTAL_EVENTS = 120000
TOTAL_USERS = 10000

PRODUCT_IDS = list(range(1000, 5000))

LOCATIONS = [
    "Delhi",
    "Mumbai",
    "Bangalore",
    "Hyderabad",
    "Pune",
    "Chennai",
    "Kolkata"
]

DEVICE_TYPES = [
    "mobile",
    "desktop",
    "tablet"
]

# ------------------------------------------
# SESSION PATTERNS
# ------------------------------------------

NORMAL_PATTERNS = [

    ['view', 'view', 'cart', 'purchase'],

    ['view', 'cart', 'purchase'],

    ['view', 'view'],

    ['view', 'cart'],

    ['view', 'view', 'view']
]

SUSPICIOUS_PATTERNS = [

    # rapid purchase
    ['purchase',
     'purchase',
     'purchase',
     'purchase'],

    # direct purchase
    ['purchase'],

    # fast checkout
    ['view',
     'purchase',
     'purchase']
]

# ------------------------------------------
# USER STATE MEMORY
# ------------------------------------------

user_state = {}

# ------------------------------------------
# EVENT GENERATION
# ------------------------------------------

for i in range(TOTAL_EVENTS):

    user_id = random.randint(
        1,
        TOTAL_USERS
    )

    # --------------------------------------
    # create user memory first time
    # --------------------------------------

    if user_id not in user_state:

        user_state[user_id] = {

            "session_id":
            str(uuid.uuid4()),

            "location":
            random.choice(
                LOCATIONS
            ),

            "device":
            random.choice(
                DEVICE_TYPES
            ),

            "pattern": [],

            "base_price":
            random.randint(
                300,
                5000
            ),

            "last_time":
            datetime.now()
        }

    user = user_state[user_id]

    # --------------------------------------
    # new session generation
    # --------------------------------------

    if len(user["pattern"]) == 0:

        # 95% normal session
        suspicious = (
            random.random() < 0.05
        )

        if suspicious:

            user["pattern"] = (
                random.choice(
                    SUSPICIOUS_PATTERNS
                ).copy()
            )

        else:

            user["pattern"] = (
                random.choice(
                    NORMAL_PATTERNS
                ).copy()
            )

        # new session
        user["session_id"] = (
            str(uuid.uuid4())
        )

    # --------------------------------------
    # current event
    # --------------------------------------

    event_type = (
        user["pattern"].pop(0)
    )

    # --------------------------------------
    # timestamp generation
    # --------------------------------------

    # suspicious fast activity
    if random.random() < 0.05:

        seconds_gap = random.randint(
            0,
            1
        )

    else:

        seconds_gap = random.randint(
            5,
            30
        )

    current_time = (

        user["last_time"]

        +

        timedelta(
            seconds=seconds_gap
        )

    )

    user["last_time"] = (
        current_time
    )

    # --------------------------------------
    # price
    # --------------------------------------

    base_price = (
        user["base_price"]
    )

    price = random.randint(

        max(100, base_price - 500),

        base_price + 500

    )

    # rare price spike
    if (
        event_type == "purchase"
        and
        random.random() < 0.03
    ):

        price *= random.randint(
            5,
            10
        )

    # --------------------------------------
    # login status
    # --------------------------------------

    is_logged_in = True

    # only suspicious purchase
    if (
        event_type == "purchase"
        and
        random.random() < 0.04
    ):

        is_logged_in = False

    # --------------------------------------
    # location logic
    # --------------------------------------

    location = (
        user["location"]
    )

    # rare location change
    if (
        event_type == "purchase"
        and
        random.random() < 0.03
    ):

        location = random.choice(
            LOCATIONS
        )

    user["location"] = (
        location
    )

    # --------------------------------------
    # event dictionary
    # --------------------------------------

    event = {

        "event_id":
        str(uuid.uuid4()),

        "timestamp":
        current_time.isoformat(),

        "user_id":
        user_id,

        "session_id":
        user["session_id"],

        "product_id":
        random.choice(
            PRODUCT_IDS
        ),

        "event_type":
        event_type,

        "price":
        price,

        "location":
        location,

        "device_type":
        user["device"],

        "is_logged_in":
        is_logged_in
    }

    # --------------------------------------
    # send to kafka
    # --------------------------------------

    producer.send(
        'user_events',
        value=event
    )

    # progress
    if i % 500 == 0:

        print(
            f"{i} events sent..."
        )

    # simulate streaming
    time.sleep(0.005)

# ---------------------------------------------------
# FINISH
# ---------------------------------------------------

producer.flush()

print(
    "\nAll 120000 events sent successfully."
)

Producer Started...

0 events sent...
500 events sent...
1000 events sent...
1500 events sent...
2000 events sent...
2500 events sent...
3000 events sent...
3500 events sent...
4000 events sent...
4500 events sent...
5000 events sent...
5500 events sent...
6000 events sent...
6500 events sent...
7000 events sent...
7500 events sent...
8000 events sent...
8500 events sent...
9000 events sent...
9500 events sent...
10000 events sent...
10500 events sent...
11000 events sent...
11500 events sent...
12000 events sent...
12500 events sent...
13000 events sent...
13500 events sent...
14000 events sent...
14500 events sent...
15000 events sent...
15500 events sent...
16000 events sent...
16500 events sent...
17000 events sent...
17500 events sent...
18000 events sent...
18500 events sent...
19000 events sent...
19500 events sent...
20000 events sent...
20500 events sent...
21000 events sent...
21500 events sent...
22000 events sent...
22500 events sent...
23000 events sent...
23500 events sen